# Future Internet 2026: full forecasting experiment runner

This notebook runs the actual zero-shot model inference and creates the run-level RMSE/PICP CSV consumed by `future_internet_2026_reproduction_code.ipynb`. Run it preferably with a GPU. Generated files go to `reproduced_outputs/regenerated/`; the archived paper-results CSV is never overwritten.

After a complete rerun, open the companion analysis notebook and set `USE_REGENERATED = True`. A fresh run need not be bit-for-bit identical to the archived CSV if upstream packages, datasets, checkpoints, hardware, or nondeterministic GPU operations have changed.

In [ ]:
# ============================================================
# 0) SETUP
# ============================================================
import sys
import subprocess
import importlib.util

def run_pip(args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)

try:
    import torch
except Exception:
    run_pip(['torch'])

run_pip([
    'numpy', 'pandas', 'huggingface_hub', 'accelerate',
    'einops', 'safetensors', 'boto3', 'botocore'
])

try:
    import chronos
except Exception:
    run_pip(['chronos-forecasting==2.3.0'])

try:
    import timesfm
except Exception:
    run_pip(['timesfm[torch]==2.0.1'])

try:
    import tirex
except Exception:
    try:
        run_pip(['tirex-ts==1.4.2', '--extra-index-url', 'https://download.pytorch.org/whl/cu121'])
    except Exception:
        run_pip(['git+https://github.com/NX-AI/tirex.git@v1.4.2'])

In [ ]:
# ============================================================
# 1) IMPORTS, PROTOCOL, DATASETS, AND CONDITIONS
# ============================================================
import time
import math
import hashlib
import traceback
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

warnings.filterwarnings('ignore')

SEED = 42
# NumPy/PyTorch top-level seed; per-window corruption seeds use stable_seed below.
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CONTEXT_LEN = 600
HORIZON_LEN = 60
STEP_LEN = 120
LOW_Q = 0.10
HIGH_Q = 0.90
PICP_COL = 'PICP80_%'
ANOMALY_COLUMN = 'anomaly'
N_RANDOM_RUNS = 10

MODEL_ORDER = ['Chronos-2', 'TimesFM-2.5', 'TiRex']
BATCH_WINDOWS = {'Chronos-2': 4, 'TimesFM-2.5': 8, 'TiRex': 8}
REQUIRE_ALL_MODELS = True

DATASET_SPECS = [
    {
        'dataset_group': 'valve1',
        'dataset_id': 'valve1_0',
        'url': 'https://raw.githubusercontent.com/waico/SKAB/master/data/valve1/0.csv',
    },
    {
        'dataset_group': 'anomaly-free',
        'dataset_id': 'anomaly_free',
        'url': 'https://raw.githubusercontent.com/waico/SKAB/master/data/anomaly-free/anomaly-free.csv',
    },
]

TARGET_CANDIDATES = {
    'Accelerometer1RMS': ['Accelerometer1RMS'],
    'Accelerometer2RMS': ['Accelerometer2RMS'],
    'Current': ['Current'],
    'Pressure': ['Pressure'],
    'Temperature': ['Temperature'],
    'Thermocouple': ['Thermocouple'],
    'Voltage': ['Voltage'],
    'FlowRate': ['Volume Flow RateRMS', 'RateRMS', 'FlowRate'],
}

# Clean baseline plus the selected robustness conditions.
SCENARIOS = [
    {'scenario_id': '00_clean', 'scenario_name': 'Clean', 'type': 'clean', 'value': None},
    {'scenario_id': '01_rpm_5pct', 'scenario_name': 'RPM 5%', 'type': 'random_missing', 'value': 0.05},
    {'scenario_id': '02_rpm_10pct', 'scenario_name': 'RPM 10%', 'type': 'random_missing', 'value': 0.10},
    {'scenario_id': '03_rpm_20pct', 'scenario_name': 'RPM 20%', 'type': 'random_missing', 'value': 0.20},
    {'scenario_id': '04_cbm_100', 'scenario_name': 'CBM 100', 'type': 'block_missing', 'value': 100},
    {'scenario_id': '05_cbm_200', 'scenario_name': 'CBM 200', 'type': 'block_missing', 'value': 200},
    {'scenario_id': '06_cbm_400', 'scenario_name': 'CBM 400', 'type': 'block_missing', 'value': 400},
    {'scenario_id': '07_pai_2sigma_1pct', 'scenario_name': 'PAI +2σ at 1%', 'type': 'spikes', 'value': 2},
    {'scenario_id': '08_pai_3sigma_1pct', 'scenario_name': 'PAI +3σ at 1%', 'type': 'spikes', 'value': 3},
    {'scenario_id': '09_pai_5sigma_1pct', 'scenario_name': 'PAI +5σ at 1%', 'type': 'spikes', 'value': 5},
]

OUTPUT_DIR = Path('reproduced_outputs/regenerated')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_PATH = OUTPUT_DIR / 'robust_tsfms_rmse_picp_results.csv'
RAW_OUTPUT_TABLE_PATH = OUTPUT_DIR / 'robust_tsfms_rmse_picp_results_all_runs.csv'
assert LOW_Q == 0.10 and HIGH_Q == 0.90 and PICP_COL == 'PICP80_%'
assert len(SCENARIOS) == 10
assert N_RANDOM_RUNS == 10

In [ ]:
# ============================================================
# 2) METRICS, DATA, AND ROBUSTNESS HELPERS
# ============================================================
def read_csv_auto(path_or_url):
    df = pd.read_csv(path_or_url, sep=None, engine='python')
    df.columns = df.columns.astype(str).str.strip()
    return df

def resolve_target_map(df):
    target_map = {}
    for target_label, candidates in TARGET_CANDIDATES.items():
        for column in candidates:
            if column in df.columns:
                target_map[target_label] = column
                break
    return target_map

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))) if mask.any() else np.nan

def picp(y_true, q_low, q_high):
    y_true = np.asarray(y_true, dtype=np.float64)
    q_low = np.asarray(q_low, dtype=np.float64)
    q_high = np.asarray(q_high, dtype=np.float64)
    mask = np.isfinite(y_true) & np.isfinite(q_low) & np.isfinite(q_high)
    return float(np.mean((y_true[mask] >= q_low[mask]) & (y_true[mask] <= q_high[mask])) * 100.0) if mask.any() else np.nan

def numeric_col_by_quantile(df, q):
    candidates = [str(q), f'{q:.2f}', f'{q:.1f}', f'q{q}', f'q{q:.2f}', f'q{int(round(q * 100))}', f'p{int(round(q * 100))}']
    for column in candidates:
        if column in df.columns:
            return column
    for column in df.columns:
        try:
            if abs(float(column) - q) < 1e-9:
                return column
        except Exception:
            pass
    return None

def get_point_col(df):
    for column in ['predictions', 'prediction', 'mean', 'median', '0.5', '0.50', 'q0.5', 'q0.50', 'q50', 'p50']:
        if column in df.columns:
            return column
    q50 = numeric_col_by_quantile(df, 0.5)
    if q50 is not None:
        return q50
    raise KeyError(f'Could not find point prediction column. Columns={df.columns.tolist()}')

def pad_to_horizon(x, horizon):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) < horizon:
        x = np.pad(x, (0, horizon - len(x)), constant_values=np.nan)
    return x[:horizon]

def to_numpy_safe(x):
    if isinstance(x, np.ndarray):
        return x
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def nearest_quantile_index(q_count, q_level):
    if q_count == 9:
        levels = np.arange(0.1, 1.0, 0.1)
    elif q_count == 3:
        levels = np.array([0.1, 0.5, 0.9])
    else:
        levels = np.linspace(1 / (q_count + 1), q_count / (q_count + 1), q_count)
    index = int(np.argmin(np.abs(levels - q_level)))
    return index, levels[index]

def stable_seed(*parts):
    text = '|'.join(map(str, parts))
    digest = hashlib.md5(text.encode('utf-8')).hexdigest()
    return (int(digest[:8], 16) + SEED) % (2**32 - 1)

def corrupt_context(ctx, scenario, rng, anomaly_labels=None):
    # The degradation changes the input context only; the evaluation horizon is never corrupted.
    # If anomaly labels are available, original anomaly-labelled points (anomaly == 1) are protected.
    # If anomaly labels are unavailable, all points are treated as non-anomalous.
    x = np.asarray(ctx, dtype=np.float32).copy()
    if anomaly_labels is None:
        anomaly_labels = np.zeros(len(x), dtype=np.int8)
    anomaly_labels = np.asarray(anomaly_labels).reshape(-1)
    if len(anomaly_labels) != len(x):
        raise ValueError('Context and anomaly-label lengths must match.')
    is_original_anomaly = anomaly_labels == 1
    length = len(x)
    scenario_type = scenario['type']

    if scenario_type == 'clean':
        return x
    if scenario_type == 'random_missing':
        eligible_idx = np.flatnonzero((~is_original_anomaly) & np.isfinite(x))
        selected_idx = eligible_idx[rng.random(len(eligible_idx)) < float(scenario['value'])]
        x[selected_idx] = np.nan
        return x
    if scenario_type == 'block_missing':
        block_size = int(scenario['value'])
        start = int(rng.integers(0, length - block_size + 1))
        block_idx = np.arange(start, start + block_size)
        eligible_idx = block_idx[(~is_original_anomaly[block_idx]) & np.isfinite(x[block_idx])]
        x[eligible_idx] = np.nan
        return x
    if scenario_type == 'spikes':
        finite = x[np.isfinite(x)]
        sigma = float(np.nanstd(finite)) if len(finite) else 0.0
        if sigma == 0.0:
            return x
        eligible_idx = np.flatnonzero((~is_original_anomaly) & np.isfinite(x))
        n_spikes = min(int(math.ceil(0.01 * len(eligible_idx))), len(eligible_idx))
        if n_spikes == 0:
            return x
        spike_idx = rng.choice(eligible_idx, size=n_spikes, replace=False)
        x[spike_idx] = x[spike_idx] + float(scenario['value']) * sigma
        return x
    raise ValueError(f"Unknown scenario type: {scenario_type}")

In [ ]:
# ============================================================
# 3) LOAD SKAB DATASETS
# ============================================================
from io import BytesIO
from urllib.request import urlopen

loaded_datasets = []
for spec in DATASET_SPECS:
    source_bytes = urlopen(spec['url']).read()
    df = read_csv_auto(BytesIO(source_bytes))
    source_sha256 = hashlib.sha256(source_bytes).hexdigest()
    target_map = resolve_target_map(df)
    if ANOMALY_COLUMN in df.columns:
        anomaly_labels = pd.to_numeric(df[ANOMALY_COLUMN], errors='coerce').fillna(0).to_numpy()
        has_anomaly_column = True
    else:
        # Anomaly-free datasets have no anomaly labels; treat every point as non-anomalous.
        anomaly_labels = np.zeros(len(df), dtype=np.int8)
        has_anomaly_column = False
    if len(target_map) != len(TARGET_CANDIDATES):
        raise RuntimeError(
            f"Expected {len(TARGET_CANDIDATES)} targets for {spec['dataset_id']}, "
            f"found {len(target_map)}: {sorted(target_map)}"
        )
    loaded_datasets.append({
        'dataset_group': spec['dataset_group'],
        'dataset_id': spec['dataset_id'],
        'df': df,
        'source_url': spec['url'],
        'source_sha256': source_sha256,
        'anomaly_labels': anomaly_labels,
        'has_anomaly_column': has_anomaly_column,
        'target_map': target_map,
    })

if not loaded_datasets:
    raise RuntimeError('No datasets loaded.')

dataset_summary = pd.DataFrame([
    {
        'dataset_group': d['dataset_group'],
        'dataset_id': d['dataset_id'],
        'n_rows': len(d['df']),
        'n_targets': len(d['target_map']),
        'targets': list(d['target_map'].keys()),
        'has_anomaly_column': d['has_anomaly_column'],
        'source_sha256': d['source_sha256'],
    }
    for d in loaded_datasets
])
display(dataset_summary)

In [ ]:
# ============================================================
# 4) LOAD MODELS — retained from the provided runner
# ============================================================
models = {}
loaded_checkpoint_ids = {}
model_errors = []

try:
    from chronos import Chronos2Pipeline
    device_map = 'cuda' if DEVICE == 'cuda' else 'cpu'
    try:
        chronos_model = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map=device_map)
        loaded_checkpoint_ids['Chronos-2'] = 'amazon/chronos-2'
    except Exception:
        chronos_model = Chronos2Pipeline.from_pretrained('s3://autogluon/chronos-2', device_map=device_map)
        loaded_checkpoint_ids['Chronos-2'] = 's3://autogluon/chronos-2'
    models['Chronos-2'] = chronos_model
except Exception as exc:
    model_errors.append({'model': 'Chronos-2', 'error': str(exc), 'traceback': traceback.format_exc()})

try:
    import timesfm
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass
    TFM_CLASS = timesfm.TimesFM_2p5_200M_torch
    if not hasattr(TFM_CLASS, '_original_from_pretrained_no_proxy_patch'):
        TFM_CLASS._original_from_pretrained_no_proxy_patch = TFM_CLASS._from_pretrained.__func__
    def _patched_from_pretrained(cls, *args, **kwargs):
        kwargs.pop('proxies', None)
        kwargs.pop('resume_download', None)
        return cls._original_from_pretrained_no_proxy_patch(cls, *args, **kwargs)
    TFM_CLASS._from_pretrained = classmethod(_patched_from_pretrained)
    try:
        timesfm_model = TFM_CLASS.from_pretrained('google/timesfm-2.5-200m-pytorch', torch_compile=False)
    except TypeError:
        timesfm_model = TFM_CLASS.from_pretrained('google/timesfm-2.5-200m-pytorch')
    try:
        timesfm_model.to(DEVICE)
    except Exception:
        pass
    timesfm_model.compile(timesfm.ForecastConfig(
        max_context=max(CONTEXT_LEN, 1024),
        max_horizon=max(HORIZON_LEN, 256),
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    ))
    models['TimesFM-2.5'] = timesfm_model
    loaded_checkpoint_ids['TimesFM-2.5'] = 'google/timesfm-2.5-200m-pytorch'
except Exception as exc:
    model_errors.append({'model': 'TimesFM-2.5', 'error': str(exc), 'traceback': traceback.format_exc()})

try:
    from tirex import load_model
    models['TiRex'] = load_model('NX-AI/TiRex', backend='torch')
    loaded_checkpoint_ids['TiRex'] = 'NX-AI/TiRex'
except Exception as exc:
    model_errors.append({'model': 'TiRex', 'error': str(exc), 'traceback': traceback.format_exc()})

if REQUIRE_ALL_MODELS:
    missing = [name for name in MODEL_ORDER if name not in models]
    if missing:
        raise RuntimeError(f'Missing required models: {missing}. Details: {model_errors}')

## Reproduction manifest

In [ ]:
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

def installed_version(distribution):
    try:
        return version(distribution)
    except PackageNotFoundError:
        return None

def git_head():
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return None

hf_heads = {}
try:
    from huggingface_hub import HfApi
    api = HfApi()
    for checkpoint in loaded_checkpoint_ids.values():
        if not checkpoint.startswith('s3://'):
            try:
                hf_heads[checkpoint] = api.model_info(checkpoint).sha
            except Exception as exc:
                hf_heads[checkpoint] = f'unavailable: {exc}'
except Exception as exc:
    hf_heads['lookup_error'] = str(exc)

manifest = {
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'code_git_commit_if_available': git_head(),
    'runner_notebook_sha256_if_available': (
        hashlib.sha256(Path('robust_tsfms_experiment_runner.ipynb').read_bytes()).hexdigest()
        if Path('robust_tsfms_experiment_runner.ipynb').is_file() else None
    ),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'device': DEVICE,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
    'torch_version': torch.__version__,
    'torch_cuda_version': torch.version.cuda,
    'packages': {name: installed_version(name) for name in [
        'numpy', 'pandas', 'torch', 'chronos-forecasting', 'timesfm', 'tirex-ts',
        'huggingface-hub', 'accelerate', 'einops', 'safetensors', 'boto3', 'botocore'
    ]},
    'loaded_checkpoint_ids': loaded_checkpoint_ids,
    'observed_hf_head_sha_not_pinned': hf_heads,
    'datasets': [{
        'dataset_id': d['dataset_id'], 'url': d['source_url'],
        'raw_file_sha256': d['source_sha256'], 'rows': len(d['df']),
        'targets': d['target_map'], 'has_anomaly_column': d['has_anomaly_column']
    } for d in loaded_datasets],
    'settings': {
        'seed': SEED,
        'per_window_seed': 'md5(dataset_id|target|scenario_type|scenario_value|start_idx|run_id); '
                           'first 8 hex digits plus 42, modulo 2**32-1',
        'context_length': CONTEXT_LEN, 'horizon_length': HORIZON_LEN,
        'window_stride': STEP_LEN, 'quantiles': [LOW_Q, 0.5, HIGH_Q],
        'random_runs_per_corruption': N_RANDOM_RUNS,
        'model_batch_windows': BATCH_WINDOWS,
        'scenarios': SCENARIOS,
        'timesfm_forecast_config': {
            'max_context': max(CONTEXT_LEN, 1024),
            'max_horizon': max(HORIZON_LEN, 256),
            'normalize_inputs': True,
            'use_continuous_quantile_head': True,
            'force_flip_invariance': True,
            'infer_is_positive': True,
            'fix_quantile_crossing': True,
            'torch_compile': False
        },
        'native_missing_values_preserved': True,
        'protect_anomaly_label_1_in_context': True,
        'forecast_horizon_is_clean': True
    }
}
manifest_path = OUTPUT_DIR / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
(OUTPUT_DIR / 'pip_freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True),
    encoding='utf-8'
)
print('Saved', manifest_path, 'and', OUTPUT_DIR / 'pip_freeze.txt')
display({key: manifest[key] for key in ['packages', 'loaded_checkpoint_ids', 'datasets', 'settings']})

In [ ]:
# ============================================================
# 5) PREDICTION HELPERS
# ============================================================
def predict_chronos_batch(model, contexts, horizon):
    fake_ts = pd.date_range(pd.Timestamp('2000-01-01'), periods=len(contexts[0]), freq='s')
    ctx_df = pd.concat([pd.DataFrame({'id': f'w{i}', 'timestamp': fake_ts, 'target': np.asarray(ctx, dtype=np.float32)}) for i, ctx in enumerate(contexts)], ignore_index=True)
    try:
        out = model.predict_df(ctx_df, prediction_length=horizon, quantile_levels=sorted({LOW_Q, 0.5, HIGH_Q}), id_column='id', timestamp_column='timestamp', target='target')
    except TypeError:
        out = model.predict_df(ctx_df, prediction_length=horizon, quantile_levels=sorted({LOW_Q, 0.5, HIGH_Q}), id_column='id', timestamp_column='timestamp', target_column='target')
    point_list, qlo_list, qhi_list = [], [], []
    for i in range(len(contexts)):
        rows = out[out['id'] == f'w{i}']
        if rows.empty:
            point_list.append(np.full(horizon, np.nan, dtype=np.float32))
            qlo_list.append(np.full(horizon, np.nan, dtype=np.float32))
            qhi_list.append(np.full(horizon, np.nan, dtype=np.float32))
            continue
        point_col = get_point_col(rows)
        lo_col = numeric_col_by_quantile(rows, LOW_Q)
        hi_col = numeric_col_by_quantile(rows, HIGH_Q)
        point_list.append(pad_to_horizon(rows[point_col].to_numpy(), horizon))
        qlo_list.append(pad_to_horizon(rows[lo_col].to_numpy(), horizon) if lo_col else np.full(horizon, np.nan, dtype=np.float32))
        qhi_list.append(pad_to_horizon(rows[hi_col].to_numpy(), horizon) if hi_col else np.full(horizon, np.nan, dtype=np.float32))
    return np.asarray(point_list), np.asarray(qlo_list), np.asarray(qhi_list)

def predict_timesfm_batch(model, contexts, horizon):
    point_fcst, quant_fcst = model.forecast(horizon=horizon, inputs=[np.asarray(ctx, dtype=np.float32) for ctx in contexts])
    point = np.asarray(point_fcst, dtype=np.float32)
    if point.ndim == 1:
        point = point[None, :]
    if point.ndim == 3:
        point = point.squeeze(-1)
    point = point[:, :horizon]
    qlo = np.full_like(point, np.nan, dtype=np.float32)
    qhi = np.full_like(point, np.nan, dtype=np.float32)
    if quant_fcst is not None:
        q = np.asarray(quant_fcst, dtype=np.float32)
        if q.ndim == 2:
            q = q[None, :, :]
        if q.ndim == 3 and q.shape[1] < horizon and q.shape[2] >= horizon:
            q = np.transpose(q, (0, 2, 1))
        if q.ndim == 3 and q.shape[1] >= horizon:
            # TimesFM 2.5 exposes [mean, q10, ..., q90]; retain support for a q10...q90-only tensor too.
            if q.shape[2] == 10:
                qlo, qhi = q[:, :horizon, 1], q[:, :horizon, 9]
            else:
                lo_idx, _ = nearest_quantile_index(q.shape[2], LOW_Q)
                hi_idx, _ = nearest_quantile_index(q.shape[2], HIGH_Q)
                qlo, qhi = q[:, :horizon, lo_idx], q[:, :horizon, hi_idx]
    return point, qlo, qhi

def extract_tirex_quantile(q_arr, q_level, horizon):
    q = to_numpy_safe(q_arr).astype(np.float32)
    if q.ndim == 2:
        q = q[None, :, :] if q.shape[0] == horizon else np.transpose(q, (1, 0))[None, :, :]
    if q.ndim != 3:
        return None
    if q.shape[1] != horizon and q.shape[2] == horizon:
        q = np.transpose(q, (0, 2, 1))
    if q.shape[1] < horizon:
        return None
    index, level = nearest_quantile_index(q.shape[2], q_level)
    return q[:, :horizon, index] if abs(level - q_level) <= 0.05 else None

def predict_tirex_batch(model, contexts, horizon):
    context = torch.from_numpy(np.stack(contexts).astype(np.float32))
    if DEVICE == 'cuda':
        context = context.cuda()
    with torch.no_grad():
        output = model.forecast(context=context, prediction_length=horizon)
    if not isinstance(output, tuple) or len(output) != 2:
        raise RuntimeError('Unexpected TiRex output format.')
    first, second = output
    quantiles, mean = (first, second) if to_numpy_safe(first).ndim >= 3 else (second, first)
    point = to_numpy_safe(mean).astype(np.float32)
    if point.ndim == 1:
        point = point[None, :]
    if point.ndim == 3:
        point = point.squeeze(-1)
    point = point[:, :horizon]
    qlo = extract_tirex_quantile(quantiles, LOW_Q, horizon)
    qhi = extract_tirex_quantile(quantiles, HIGH_Q, horizon)
    if qlo is None:
        qlo = np.full_like(point, np.nan, dtype=np.float32)
    if qhi is None:
        qhi = np.full_like(point, np.nan, dtype=np.float32)
    return point, qlo, qhi

def predict_batch(model_name, model, contexts, horizon):
    if model_name == 'Chronos-2':
        return predict_chronos_batch(model, contexts, horizon)
    if model_name == 'TimesFM-2.5':
        return predict_timesfm_batch(model, contexts, horizon)
    if model_name == 'TiRex':
        return predict_tirex_batch(model, contexts, horizon)
    raise ValueError(f'Unknown model: {model_name}')

## Execute the complete experiment

In [ ]:
# ============================================================
# 6) RUN EXPERIMENTS AND GENERATE AVERAGED + RUN-LEVEL RESULTS
# ============================================================
def run_one_combination(dataset_info, target_label, actual_col, scenario, run_id, model_name, model):
    y = pd.to_numeric(dataset_info['df'][actual_col], errors='coerce').to_numpy(dtype=np.float32)
    if len(y) < CONTEXT_LEN + HORIZON_LEN:
        raise ValueError(f"Series too short: {dataset_info['dataset_id']} / {target_label}")

    actual_parts, prediction_parts, qlow_parts, qhigh_parts = [], [], [], []
    n_windows = 0
    starts = list(range(CONTEXT_LEN, len(y) - HORIZON_LEN + 1, STEP_LEN))
    batch_size = BATCH_WINDOWS.get(model_name, 4)

    for batch_start in range(0, len(starts), batch_size):
        contexts, horizons = [], []
        for start_idx in starts[batch_start:batch_start + batch_size]:
            clean_context = y[start_idx - CONTEXT_LEN:start_idx]
            anomaly_context = dataset_info['anomaly_labels'][start_idx - CONTEXT_LEN:start_idx]
            true_horizon = y[start_idx:start_idx + HORIZON_LEN]
            if not np.isfinite(true_horizon).any():
                continue
            rng = np.random.default_rng(stable_seed(
                dataset_info['dataset_id'],
                target_label,
                scenario['type'],
                scenario['value'],
                start_idx,
                run_id,
            ))
            contexts.append(corrupt_context(clean_context, scenario, rng, anomaly_context))
            horizons.append(true_horizon)

        if not contexts:
            continue

        point_batch, qlo_batch, qhi_batch = predict_batch(model_name, model, contexts, HORIZON_LEN)
        for i, true_horizon in enumerate(horizons):
            actual_parts.append(true_horizon)
            prediction_parts.append(pad_to_horizon(point_batch[i], HORIZON_LEN))
            qlow_parts.append(pad_to_horizon(qlo_batch[i], HORIZON_LEN))
            qhigh_parts.append(pad_to_horizon(qhi_batch[i], HORIZON_LEN))
            n_windows += 1

    if not actual_parts:
        raise RuntimeError('No predictions were produced.')

    y_true = np.concatenate(actual_parts)
    y_pred = np.concatenate(prediction_parts)
    q_low = np.concatenate(qlow_parts)
    q_high = np.concatenate(qhigh_parts)
    return {
        'dataset_id': dataset_info['dataset_id'],
        'target': target_label,
        'scenario_id': scenario['scenario_id'],
        'condition': scenario['scenario_name'],
        'run_id': run_id,
        'model': model_name,
        'RMSE': rmse(y_true, y_pred),
        PICP_COL: picp(y_true, q_low, q_high),
        'n_windows': n_windows,
    }

metric_rows, error_rows = [], []
for dataset_info in loaded_datasets:
    for target_label, actual_col in dataset_info['target_map'].items():
        for scenario in SCENARIOS:
            run_ids = [0] if scenario['type'] == 'clean' else range(N_RANDOM_RUNS)
            for run_id in run_ids:
                for model_name in MODEL_ORDER:
                    try:
                        metric_rows.append(run_one_combination(
                            dataset_info, target_label, actual_col, scenario, run_id, model_name, models[model_name]
                        ))
                    except Exception as exc:
                        error_rows.append({
                            'dataset': dataset_info['dataset_id'],
                            'target': target_label,
                            'condition': scenario['scenario_name'],
                            'run_id': run_id,
                            'model': model_name,
                            'error': str(exc),
                        })

if not metric_rows:
    raise RuntimeError(f'No metrics produced. Errors: {error_rows}')
if error_rows:
    raise RuntimeError(f'One or more experiment combinations failed; no partial table was produced. Errors: {error_rows}')

scenario_rank = {scenario['scenario_id']: rank for rank, scenario in enumerate(SCENARIOS)}
model_rank = {model_name: rank for rank, model_name in enumerate(MODEL_ORDER)}
raw_metric_table = pd.DataFrame(metric_rows).assign(
    _scenario_rank=lambda df: df['scenario_id'].map(scenario_rank),
    _model_rank=lambda df: df['model'].map(model_rank),
)

raw_result_table = raw_metric_table.sort_values(
    ['dataset_id', 'target', '_scenario_rank', 'run_id', '_model_rank']
).rename(columns={
    'dataset_id': 'Dataset',
    'target': 'Target',
    'condition': 'Condition',
    'run_id': 'Run',
    'model': 'Model',
})[['Dataset', 'Target', 'Condition', 'Run', 'Model', 'RMSE', PICP_COL, 'n_windows']].reset_index(drop=True)

result_table = raw_metric_table.groupby(
    ['dataset_id', 'target', 'scenario_id', 'condition', 'model'],
    as_index=False,
).agg(
    Runs=('run_id', 'nunique'),
    RMSE=('RMSE', 'mean'),
    **{PICP_COL: (PICP_COL, 'mean')},
).assign(
    _scenario_rank=lambda df: df['scenario_id'].map(scenario_rank),
    _model_rank=lambda df: df['model'].map(model_rank),
).sort_values(['dataset_id', 'target', '_scenario_rank', '_model_rank']).rename(columns={
    'dataset_id': 'Dataset',
    'target': 'Target',
    'condition': 'Condition',
    'model': 'Model',
})[['Dataset', 'Target', 'Condition', 'Model', 'Runs', 'RMSE', PICP_COL]].reset_index(drop=True)

raw_result_table.to_csv(RAW_OUTPUT_TABLE_PATH, index=False)
result_table.to_csv(OUTPUT_TABLE_PATH, index=False)
display(result_table)

## Continue to tables and figures

After the experiment completes, leave the generated CSV at `reproduced_outputs/regenerated/robust_tsfms_rmse_picp_results_all_runs.csv`, relative to the directory containing `future_internet_2026_reproduction_code.ipynb`, and set `USE_REGENERATED = True` in that notebook. The analysis notebook validates the row grid, then rebuilds Table 2, both Anomaly-Free figures, the cross-dataset diagnostic, and the Valve 1 representative-run table.